# The scalar effective potential, end to end

This notebook runs a $\phi^4$ effective-potential calculation through Phaser's
public Python API: it loads a model, derives the tree and one-loop artifacts,
renders their equations, binds two parameter points, evaluates single points and
batches, checks a gradient against an independent reference, and plots the
result.

Everything here uses public interfaces only. No physics, no numerics, and no
validation happen in this document; every number comes from the same core the
command-line client and the C conformance client use.

## Running it

Build the binding first, then start Jupyter with it on the path:

```
zig build python -Doptimize=ReleaseSafe -Dpython=$(command -v python3)
PYTHONPATH=zig-out/python jupyter lab docs/notebooks/
```

Outputs are not committed. Run the notebook to see them, and clear them again
before committing:

```
python3 tools/ci/clear_notebook_outputs.py docs/notebooks/scalar_effective_potential.ipynb
```

Continuous integration executes this notebook from a fresh kernel on every pull
request and fails the build if any cell raises.

**The ABI is experimental.** Version 0 permits breaking changes, and this
notebook tracks them rather than pinning against them.

In [ ]:
import os
import pathlib

import matplotlib.pyplot as plt

import phaser

print("Phaser library version:", ".".join(str(part) for part in phaser.library_version()))
print("C ABI version:", phaser.abi_version(), "(experimental)" if phaser.abi_experimental() else "")

The committed example inputs live in `examples/phi4/`. Locating them by walking
up from the working directory keeps this notebook runnable from a checkout
without any environment setup; `PHASER_EXAMPLES` overrides it, which is what
continuous integration uses.

In [ ]:
def find_examples():
    override = os.environ.get("PHASER_EXAMPLES")
    if override:
        return pathlib.Path(override) / "phi4"
    here = pathlib.Path.cwd().resolve()
    for directory in [here, *here.parents]:
        candidate = directory / "examples" / "phi4"
        if candidate.is_dir():
            return candidate
    raise RuntimeError("could not locate examples/phi4 from " + str(here))


EXAMPLES = find_examples()
print("Reading committed inputs from", EXAMPLES)

## 1. The model

A single real scalar in four dimensions with a mass term, a quartic coupling,
and a vacuum-energy constant. The conventions are part of the document rather
than assumed: mostly-plus metric, real scalar components, two-component Weyl
fermions.

In [ ]:
model_source = (EXAMPLES / "model.json").read_bytes()
print(model_source.decode())

In [ ]:
context = phaser.Context()
model = phaser.Model(model_source, context=context)

print("fingerprint       ", model.fingerprint)
print("parameters        ", model.parameter_count)
print("real scalar fields", model.scalar_field_count)

The fingerprint identifies the canonical model rather than its source text: a
reformatted document has the same fingerprint. Every result below belongs to
this fingerprint, and quoting it is how a plot is traced back to an input.

## 2. The calculation

Two requests over the same model. The first truncates at tree level, the second
at one loop; both vary the full scalar background and renormalize in
$\overline{\text{MS}}$.

In [ ]:
tree_request = phaser.Request((EXAMPLES / "request.json").read_bytes(), context=context)
one_loop_request = phaser.Request(
    (EXAMPLES / "request_one_loop.json").read_bytes(), context=context
)

tree = model.derive(tree_request)
one_loop = model.derive(one_loop_request)

print("tree    ", tree)
print("one loop", one_loop)

## 3. The equations

Displaying an artifact renders its equations through the MathJax-compatible
LaTeX exporter. This is the same fragment the `export("latex")` call returns;
the notebook frontend supplies the delimiters.

In [ ]:
tree

In [ ]:
one_loop

Outside a notebook the same objects print as plain text, which is what a
terminal or a log gets:

In [ ]:
print(tree)

## 4. Metadata

Loop order, contribution count, and result type are typed queries rather than
text to be parsed. The result type is the one that matters most here: a
tree-level potential is real, and a loop-containing one is not.

In [ ]:
header = f"{'':10}{'loop order':>12}{'coordinates':>13}{'contributions':>15}{'result':>12}"
print(header)
for name, artifact in (("tree", tree), ("one loop", one_loop)):
    print(
        f"{name:10}{artifact.loop_order:>12}{artifact.coordinate_count:>13}"
        f"{artifact.contribution_count:>15}{artifact.result_type:>12}"
    )

## 5. Two parameter points

A point carries values for every model parameter, a renormalization scheme, and
a reference scale. The same kernel binds more than one of them, and binding is
where the parameter-dependent work happens, so a scan over points pays for it
once per point rather than once per evaluation.

The second point below is written here rather than read from disk, to show that
a point is an ordinary document a caller can construct.

In [ ]:
committed_point = phaser.ParameterPoint(
    (EXAMPLES / "point.json").read_bytes(), context=context
)

lighter_source = b'''{
  "schema": "phaser.parameter-point/0.1",
  "units": { "mass": "GeV" },
  "renormalization": { "scheme": "MSbar", "reference_scale": 125.0 },
  "values": { "lambda": 0.52, "m2": -7812.5, "omega": 0 }
}'''
lighter_point = phaser.ParameterPoint(lighter_source, context=context)

print("committed point, reference scale", committed_point.reference_scale, "GeV")
print("second point,    reference scale", lighter_point.reference_scale, "GeV")

In [ ]:
tree_kernel = tree.compile()
one_loop_kernel = one_loop.compile()

bindings = {
    "lambda = 0.26": one_loop_kernel.bind(committed_point),
    "lambda = 0.52": one_loop_kernel.bind(lighter_point),
}
tree_binding = tree_kernel.bind(committed_point)

for label, binding in bindings.items():
    print(f"{label}: {binding}")

Doubling the quartic coupling halves the squared field value at which the
field-dependent mass-squared $m^2 + \tfrac{1}{2}\lambda\phi^2$ changes sign, so
the two bindings should differ in where their imaginary part vanishes. That is
the feature to look for in the second plot.

## 6. Scalar and batch evaluation

The same kernel answers one point and a whole array. A batch crosses through the
buffer protocol, so an `array.array('d')` or a NumPy array is read without a
copy; a plain list is copied into one.

These must agree exactly, not approximately. Points in a batch are independent.

In [ ]:
GRID = [float(50 * step) for step in range(13)]  # 0 to 600 GeV
print("grid:", GRID)

binding = bindings["lambda = 0.26"]
batch = binding.evaluate(GRID)
single = binding.evaluate_at(250.0)

index = GRID.index(250.0)
print()
print("scalar call:", single.value)
print("in a batch :", batch.value(index))
print("identical  :", single == batch.point(index))

## 7. A gradient against an independent reference

The tree potential has a closed form,

$$V^{(0)}(\phi) = \Omega + \tfrac{1}{2} m^2 \phi^2 + \tfrac{1}{24}\lambda\phi^4,
\qquad
\frac{dV^{(0)}}{d\phi} = m^2 \phi + \tfrac{1}{6}\lambda\phi^3,$$

so the gradient the kernel computes can be checked against arithmetic written
out here. That is one independent reference. The second is a central difference
of the kernel's own values, which needs no closed form and so also works where
none exists.

The difference is compared under `finite_difference_gradient_well_conditioned`
from [Numerical Comparison](../architecture/NUMERICAL_COMPARISON.md): relative
tolerance `1e-8`, step $\epsilon^{1/3}\max(|\phi|, 1)$, and ten times the
roundoff the reference inherits. Phaser declares no universal tolerance, so the
policy is named rather than a literal chosen here.

In [ ]:
LAMBDA = 0.26
M2 = -7812.5
EPS = 2.0 ** -52


def analytic_tree_gradient(phi):
    return M2 * phi + LAMBDA * phi**3 / 6.0


def central_difference(binding, phi):
    # Returns the difference quotient and the roundoff it inherits.
    step = EPS ** (1.0 / 3.0) * max(abs(phi), 1.0)
    plus = binding.evaluate_at(phi + step).value
    minus = binding.evaluate_at(phi - step).value
    quotient = (plus - minus) / (2.0 * step)
    inherited = EPS * max(abs(plus), abs(minus)) / step
    return quotient, inherited


print(f"{'phi':>6}{'kernel':>22}{'closed form':>22}{'central difference':>22}")
for phi in [50.0, 100.0, 250.0, 500.0, 600.0]:
    exact = tree_binding.evaluate_at(phi).gradient[0]
    closed = analytic_tree_gradient(phi)
    quotient, inherited = central_difference(tree_binding, phi)
    print(f"{phi:>6.0f}{exact:>22.12g}{closed:>22.12g}{quotient:>22.12g}")

    budget = 1e-8 * max(abs(exact), abs(quotient)) + 10.0 * inherited
    assert abs(exact - quotient) <= budget, (phi, exact, quotient, budget)
print()
print("Every point satisfies finite_difference_gradient_well_conditioned.")

The closed form and the kernel agree to the last few digits, and the central
difference agrees to the accuracy a difference quotient can offer. The middle
column is the one to read: it is arithmetic anyone can check by hand.

## 8. The potential over a background interval

Three curves on the grid above:

- the **tree** potential, from the loop-order-zero request;
- the **total** through one loop, from the loop-order-one request; and
- the **one-loop contribution**, as the difference of the two.

Version 0 of the C ABI has no contribution-selection operation, so the loop
piece is obtained by subtraction rather than asked for directly. The tree and
the total are compared bitwise against committed command-line output by the
machine tests in `bindings/python/test/test_extension.py`; the grid above is
exactly the one those tests assert.

In [ ]:
tree_values = tree_binding.evaluate(GRID)
total_values = binding.evaluate(GRID)

tree_curve = [tree_values.value(i) for i in range(len(GRID))]
total_curve = [total_values.value(i) for i in range(len(GRID))]
loop_curve = [total_curve[i] - tree_curve[i] for i in range(len(GRID))]

statuses = {tree_values.status(i) for i in range(len(GRID))}
statuses |= {total_values.status(i) for i in range(len(GRID))}
print("point statuses across the grid:", statuses)

In [ ]:
figure, (upper, lower) = plt.subplots(2, 1, figsize=(8, 8), sharex=True)

scale = 1e9
upper.plot(GRID, [value / scale for value in tree_curve], "o-", label="tree")
upper.plot(GRID, [value.real / scale for value in loop_curve], "s-", label="one loop (Re)")
upper.plot(GRID, [value.real / scale for value in total_curve], "^-", label="total (Re)")
upper.axhline(0.0, linewidth=0.8, color="0.6")
upper.set_ylabel(r"$V\ /\ 10^{9}\ \mathrm{GeV}^4$")
upper.legend()
upper.set_title(r"Effective potential of a single real scalar, $\overline{MS}$ at 125 GeV")

lower.plot(GRID, [value.imag / scale for value in total_curve], "^-", color="tab:red",
           label="total (Im)")
lower.axhline(0.0, linewidth=0.8, color="0.6")
lower.set_xlabel(r"$\phi\ /\ \mathrm{GeV}$")
lower.set_ylabel(r"$\mathrm{Im}\,V\ /\ 10^{9}\ \mathrm{GeV}^4$")
lower.legend()

figure.tight_layout()

The same total for both parameter points, to show where the coupling moves the
imaginary part:

In [ ]:
figure, axes = plt.subplots(figsize=(8, 4))
for label, other in bindings.items():
    results = other.evaluate(GRID)
    axes.plot(
        GRID,
        [results.value(i).imag / 1e9 for i in range(len(GRID))],
        "o-",
        label=label,
    )
axes.axhline(0.0, linewidth=0.8, color="0.6")
axes.set_xlabel(r"$\phi\ /\ \mathrm{GeV}$")
axes.set_ylabel(r"$\mathrm{Im}\,V\ /\ 10^{9}\ \mathrm{GeV}^4$")
axes.set_title("Where the field-dependent mass-squared changes sign")
axes.legend()
figure.tight_layout()

## 9. What to inspect

Specific, checkable claims about the figures above. If one of them fails to
hold, the notebook is reporting a change worth investigating rather than a
rendering quirk.

**The tree curve.** Negative $m^2$ with positive $\lambda$ gives a double well.
Along $\phi \ge 0$ the curve falls from zero, reaches a minimum where
$m^2\phi + \tfrac{1}{6}\lambda\phi^3 = 0$, that is at
$\phi = \sqrt{-6m^2/\lambda} \approx 424.6\ \mathrm{GeV}$, and rises steeply
after it. On this grid the lowest *sampled* point is $\phi = 400$, one step
short of the true minimum. It is real everywhere.

It then returns through zero: $V^{(0)}$ vanishes again at
$\phi = \sqrt{-12m^2/\lambda} \approx 600.5\ \mathrm{GeV}$, just past the right
edge of the interval. That is why the tree curve is close to zero at
$\phi = 600$ having been at $-3.5 \times 10^{8}$ near its minimum.

**The imaginary part.** It is nonzero at small $\phi$ and exactly zero at large
$\phi$, switching where the field-dependent mass-squared
$m^2 + \tfrac{1}{2}\lambda\phi^2$ passes through zero, at
$\phi = \sqrt{-2m^2/\lambda} \approx 245\ \mathrm{GeV}$. Exactly zero, not
small: above the crossing the logarithm's argument is positive and the result is
real.

A negative eigenvalue is a physical result here, not a failure. Every point in
the grid reports status `ok`, which the cell above prints.

**The second figure.** Doubling $\lambda$ moves the crossing down by a factor of
$\sqrt{2}$, to about $173.3\ \mathrm{GeV}$. On this grid the `lambda = 0.52`
curve therefore reaches zero at $\phi = 200$ and the `lambda = 0.26` curve at
$\phi = 250$ — one grid step earlier.

**The one-loop contribution.** Judge it where the tree is large. Near the
minimum the correction is a few percent of the tree: at $\phi = 50$ the ratio is
about $2\%$, which is what a one-loop term should look like when the expansion
is under control.

Do not read the ratio near the tree's two zeros, at $\phi = 0$ and
$\phi \approx 600.5$. There the denominator is what is small, not the numerator:
at $\phi = 600$ the ratio is about $0.63$, and it says the tree has nearly
cancelled itself rather than that the loop expansion has broken down. The
absolute one-loop curve in the first figure is smooth through both.

**What the plot is not.** These are twelve intervals over 600 GeV, so the curves
are piecewise-linear between grid points and the visible minimum is the smallest
sampled point rather than the true one. Refining the grid is one edit; it moves
the notebook off the set the machine tests assert bitwise, which is the trade.

The cell below prints the ratios quoted above, so they are read from the run
rather than taken on trust from this text.

In [ ]:
print(f"{'phi':>6}{'tree':>16}{'one loop (Re)':>16}{'|loop / tree|':>16}")
for i, phi in enumerate(GRID):
    tree_value = tree_curve[i]
    loop_value = loop_curve[i].real
    ratio = abs(loop_value / tree_value) if tree_value else float("inf")
    print(f"{phi:>6.0f}{tree_value:>16.4g}{loop_value:>16.4g}{ratio:>16.4g}")

minimum = min(range(len(GRID)), key=lambda i: tree_curve[i])
print()
print("lowest sampled tree point:", GRID[minimum], "GeV")
print("true tree minimum:        ", (-6 * M2 / LAMBDA) ** 0.5, "GeV")
print("tree returns to zero at:  ", (-12 * M2 / LAMBDA) ** 0.5, "GeV")

## 10. A rejected document

Diagnostics are structured, not just text. A rejected document raises
`SourceError`, which subclasses `ValueError`, and carries every diagnostic the
parser produced with its severity, category, and source span.

In [ ]:
try:
    phaser.Model(b'{"schema": "phaser.qft-model/0.1", "spacetime_dimension": ')
except phaser.SourceError as error:
    print("caught:", type(error).__name__)
    print("message:", error)
    print()
    for diagnostic in error.diagnostics:
        print(f"  severity {diagnostic.severity}")
        print(f"  category {diagnostic.category}")
        print(f"  span     {diagnostic.start}..{diagnostic.end} in source {diagnostic.source_id}")
        print(f"  message  {diagnostic.message}")

A calculation that a model cannot support is rejected the same way, at the point
where the two documents are combined rather than when either is parsed:

In [ ]:
mismatched = phaser.Request(
    b'''{
      "schema": "phaser.calculation/0.1",
      "kind": "effective_potential",
      "background": {
        "mode": "component_slice",
        "coordinates": [{"id": "h", "scalar": "nonexistent"}]
      },
      "environment": { "kind": "vacuum" },
      "renormalization": { "scheme": "MSbar" },
      "orders": { "loop": { "through": 1 } }
    }''',
    context=context,
)
print("the request itself parses:", mismatched)

try:
    model.derive(mismatched)
except phaser.SourceError as error:
    print("deriving against this model does not:", error)
    diagnostic = error.diagnostics[0]
    print("  category", diagnostic.category, "| span", diagnostic.start)

The second diagnostic carries no source span. That is deliberate and different
from a span at offset zero: the failure belongs to the combination of two valid
documents rather than to a location in either of them.